# NB02 — Model Training
**Project:** CNN vs ViT Localization Faithfulness in Chest X-Ray  
**Stage:** 2 — Run after NB01 is complete  

## What this notebook does
1. Mounts Drive and installs packages  
2. Sets global reproducibility seed  
3. Builds DataLoaders with augmentation  
4. Trains DenseNet121 (Classic CNN)  
5. Trains ConvNeXtV2-Tiny (Modern CNN)  
6. Runs LoRA rank sweep for Swin-B  
7. Trains Swin-B + LoRA (Transformer)  
8. Calibrates per-pathology thresholds  
9. Saves all weights, checkpoints, and training history  

## Before running
- NB01 must be complete ✅  
- `data/processed/images/` must have ~15,000 PNGs ✅  
- `data/processed/splits/` must have train.csv and val.csv ✅  
- **Switch runtime to GPU: Runtime → Change runtime type → T4 GPU** ✅  

## ⚠️ Run each model in a separate Colab session
Each block checks for existing checkpoint and skips if found.  
DenseNet121 → ConvNeXtV2-Tiny → Swin-B LoRA sweep → Swin-B full train.

## Batch sizes per model (plan-specified)
| Model | Batch Size | Reason |
|---|---|---|
| DenseNet121 | 32 | Standard — fits T4 comfortably |
| ConvNeXtV2-Tiny | 64 | Head-only, frozen backbone — very low VRAM |
| Swin-B + LoRA | 16 | Large model — 32 will OOM on T4 |

In [ ]:
!pip install -q "torchao>=0.16.0"
from google.colab import drive
drive.mount('/content/drive')

GDRIVE_ROOT = '/content/drive/MyDrive/cxr_faithfulness'
print("✅ Drive mounted.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 56.1 MB/s eta 0:00:00
Mounted at /content/drive
✅ Drive mounted.


In [ ]:
exec(open(f'{GDRIVE_ROOT}/config/startup.py').read())

⏳ Installing missing packages (~1 min)...
✅ Packages ready. Ignore any conflict warnings.


In [ ]:
import torch
assert torch.cuda.is_available(), "❌ GPU not available. Go to Runtime → Change runtime type → GPU"
print(f"✅ GPU ready : {torch.cuda.get_device_name(0)}")
print(f"   VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

✅ GPU ready : Tesla T4
   VRAM     : 15.6 GB


In [ ]:
import sys, os, json, random
import pandas as pd
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as tv_models
import timm
from sklearn.metrics import roc_auc_score, f1_score
from torch.cuda.amp import GradScaler, autocast
import cv2


# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT         = Path(GDRIVE_ROOT)
IMAGES_PATH  = ROOT / "data" / "processed" / "images"
SPLITS_PATH  = ROOT / "data" / "processed" / "splits"
MODELS_PATH  = ROOT / "models"
CKPT_PATH    = ROOT / "models" / "checkpoints"
RESULTS_PATH = ROOT / "results"

for p in [MODELS_PATH, CKPT_PATH, RESULTS_PATH]:
    p.mkdir(parents=True, exist_ok=True)

# ── Hyperparameters ────────────────────────────────────────────────────────────
IMG_SIZE         = 224
BATCH_SIZE       = 32   # DenseNet121 default
BATCH_SIZE_CONV  = 64   # ConvNeXtV2-Tiny: head-only, frozen backbone — low VRAM
BATCH_SIZE_SWIN  = 16   # Swin-B + LoRA: large model — 32 will OOM on T4
NUM_EPOCHS_DENSE = 15
NUM_EPOCHS_CONV  = 15
NUM_EPOCHS_SWIN  = 20
NUM_CLASSES      = 14
RANDOM_SEED      = 42

print("✅ Config loaded.")
print(f"   Batch sizes — DenseNet: {BATCH_SIZE} | ConvNeXtV2: {BATCH_SIZE_CONV} | Swin-B: {BATCH_SIZE_SWIN}")

✅ Config loaded.
   Batch sizes — DenseNet: 32 | ConvNeXtV2: 64 | Swin-B: 16


In [ ]:
import sys, os, json, random
import pandas as pd
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as tv_models
import timm
from sklearn.metrics import roc_auc_score, f1_score
from torch.cuda.amp import GradScaler, autocast
import cv2
import warnings
warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT         = Path(GDRIVE_ROOT)
IMAGES_PATH  = ROOT / "data" / "processed" / "images"
SPLITS_PATH  = ROOT / "data" / "processed" / "splits"
MODELS_PATH  = ROOT / "models"
CKPT_PATH    = ROOT / "models" / "checkpoints"
RESULTS_PATH = ROOT / "results"

for p in [MODELS_PATH, CKPT_PATH, RESULTS_PATH]:
    p.mkdir(parents=True, exist_ok=True)

# ── Hyperparameters ────────────────────────────────────────────────────────────
IMG_SIZE         = 224
BATCH_SIZE       = 32   # DenseNet121 default
BATCH_SIZE_CONV  = 64   # ConvNeXtV2-Tiny: head-only, frozen backbone — low VRAM
BATCH_SIZE_SWIN  = 16   # Swin-B + LoRA: large model — 32 will OOM on T4
NUM_EPOCHS_DENSE = 15
NUM_EPOCHS_CONV  = 15
NUM_EPOCHS_SWIN  = 20
NUM_CLASSES      = 14
RANDOM_SEED      = 42

print("✅ Config loaded.")
print(f"   Batch sizes — DenseNet: {BATCH_SIZE} | ConvNeXtV2: {BATCH_SIZE_CONV} | Swin-B: {BATCH_SIZE_SWIN}")

✅ Config loaded.
   Batch sizes — DenseNet: 32 | ConvNeXtV2: 64 | Swin-B: 16


## Step 1 — Set reproducibility seed
All 6 seed components must be set for full GPU reproducibility.  
This must run before any model init, DataLoader creation, or augmentation.

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    print(f"✅ Seed set: {seed}")

set_seed(RANDOM_SEED)

✅ Seed set: 42


## Step 2 — Dataset and DataLoaders
Augmentation on train split only.  
Val and test use normalization only — no augmentation.  
ImageNet mean/std used for all 3 models.  
`worker_init_fn` seeds each DataLoader worker for full reproducibility.  

⚠️ Three separate train loaders are created — one per model with its plan-specified batch size.  
Val loader uses batch=32 for all models (inference only — no VRAM pressure).

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

val_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

print("✅ Transforms defined.")

✅ Transforms defined.


In [ ]:
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(RANDOM_SEED)

class CXRDataset(Dataset):
    def __init__(self, csv_path, images_path, transform=None):
        self.df          = pd.read_csv(str(csv_path))
        self.images_path = Path(images_path)
        self.transform   = transform
        self.label_cols  = [c for c in self.df.columns if c != 'image_id']

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        img_p  = self.images_path / f"{row['image_id']}.png"
        img    = cv2.imread(str(img_p))
        img    = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        labels = torch.tensor(
            row[self.label_cols].values.astype(float),
            dtype=torch.float32
        )
        if self.transform:
            img = self.transform(img)
        return img, labels

# ── Datasets ───────────────────────────────────────────────────────────────────
train_dataset = CXRDataset(SPLITS_PATH / "train.csv", IMAGES_PATH, train_transforms)
val_dataset   = CXRDataset(SPLITS_PATH / "val.csv",   IMAGES_PATH, val_transforms)

LABEL_COLS = train_dataset.label_cols

# ── Model-specific train loaders (batch size differs per plan) ─────────────────
def make_train_loader(batch_size):
    return DataLoader(
        train_dataset, batch_size=batch_size,
        shuffle=True, num_workers=2, pin_memory=True,
        worker_init_fn=seed_worker, generator=g
    )

train_loader_dense = make_train_loader(BATCH_SIZE)       # 32 — DenseNet121
train_loader_conv  = make_train_loader(BATCH_SIZE_CONV)  # 64 — ConvNeXtV2-Tiny
train_loader_swin  = make_train_loader(BATCH_SIZE_SWIN)  # 16 — Swin-B + LoRA

# ── Shared val loader (batch=32 for all models — inference only) ───────────────
val_loader = DataLoader(
    val_dataset, batch_size=32,
    shuffle=False, num_workers=2, pin_memory=True,
    worker_init_fn=seed_worker, generator=g
)

print(f"✅ DataLoaders ready.")
print(f"   Train samples      : {len(train_dataset)}")
print(f"   Val samples        : {len(val_dataset)}")
print(f"   DenseNet loader    : batch={BATCH_SIZE}  → {len(train_loader_dense)} batches")
print(f"   ConvNeXtV2 loader  : batch={BATCH_SIZE_CONV}  → {len(train_loader_conv)} batches")
print(f"   Swin-B loader      : batch={BATCH_SIZE_SWIN}  → {len(train_loader_swin)} batches")
print(f"   Val batches        : {len(val_loader)}")
print(f"   Label columns      : {len(LABEL_COLS)} classes")

✅ DataLoaders ready.
   Train samples      : 12000
   Val samples        : 1500
   DenseNet loader    : batch=32  → 375 batches
   ConvNeXtV2 loader  : batch=64  → 188 batches
   Swin-B loader      : batch=16  → 750 batches
   Val batches        : 47
   Label columns      : 14 classes


## Step 3 — Training loop (shared by all 3 models)

> **Methodological Limitation:** The architecture comparison in this paper is confounded by unequal adaptation strategies. DenseNet-121 fine-tunes ~7M parameters (denseblock4 + head), whereas ConvNeXtV2-Tiny only fine-tunes its classifier head (~11K) and Swin-B utilizes LoRA (~2.3M parameters). The evaluation explicitly reframes the comparison as 'architecture + adaptation-strategy' to honestly address this varying parameter budget.

Saves every epoch checkpoint for crash recovery.  
Saves best val-loss model separately.  
Saves full training history CSV for offline figure generation.  
Seed is re-applied before each model — ensures identical initialisation across runs.

In [ ]:
def train_model(model, model_name, train_loader, val_loader,
                optimizer, scheduler, n_epochs):

    criterion = nn.BCEWithLogitsLoss()
    scaler    = GradScaler()
    best_val_loss = float('inf')
    history = []
    patience_counter = 0
    EARLY_STOP_PATIENCE = 7   # stop if no improvement for 7 epochs

    for epoch in range(1, n_epochs + 1):

        # ── Train ──────────────────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.cuda(), labels.cuda()
            optimizer.zero_grad()
            with autocast():
                outputs = model(imgs)
                loss    = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        # ── Validate ───────────────────────────────────────────────────────
        model.eval()
        val_loss   = 0.0
        all_preds  = []
        all_labels = []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.cuda(), labels.cuda()
                with autocast():
                    outputs = model(imgs)
                    loss    = criterion(outputs, labels)
                val_loss += loss.item()
                all_preds.append(torch.sigmoid(outputs).cpu().numpy())
                all_labels.append(labels.cpu().numpy())
        val_loss  /= len(val_loader)
        all_preds  = np.vstack(all_preds)
        all_labels = np.vstack(all_labels)

        try:
            val_auc_macro    = roc_auc_score(all_labels, all_preds, average='macro')
            val_auc_weighted = roc_auc_score(all_labels, all_preds, average='weighted')
            val_auc = val_auc_macro
        except ValueError:
            val_auc_macro = val_auc_weighted = 0.0
            val_auc = 0.0

        # FIX Issue 2: capture LR before and after step to detect scheduler reduction
        lr_before = optimizer.param_groups[0]['lr']
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        lr_tag = f" | lr={current_lr:.2e}"
        if current_lr < lr_before:
            lr_tag += " ↓ reduced"

        # ── Save epoch checkpoint ──────────────────────────────────────────
        ckpt = CKPT_PATH / f"{model_name}_epoch{epoch:02d}.pt"
        torch.save(model.state_dict(), str(ckpt))

        # ── Save best model ────────────────────────────────────────────────
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), str(MODELS_PATH / f'{model_name}_finetuned.pt'))
            best_tag = ' ★best'
        else:
            patience_counter += 1
            best_tag = f' (patience {patience_counter}/{EARLY_STOP_PATIENCE})'


        # ── Log — includes lr so history CSV shows scheduler behaviour ─────
        history.append({
            'epoch'      : epoch,
            'train_loss' : round(train_loss, 4),
            'val_loss'   : round(val_loss, 4),
            'val_auc'    : round(val_auc_macro, 4),
            'val_auc_w'  : round(val_auc_weighted, 4),
            'lr'         : current_lr
        })

        print(f"Epoch {epoch:02d}/{n_epochs} | "
              f"train_loss={train_loss:.4f} | "
              f"val_loss={val_loss:.4f} | "
              f"val_auc={val_auc:.4f}{lr_tag}{best_tag}")


        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f'Early stopping triggered at epoch {epoch}.')
            break

    # ── Save history CSV ───────────────────────────────────────────────────
    hist_df   = pd.DataFrame(history)
    hist_path = RESULTS_PATH / f"{model_name}_history.csv"
    hist_df.to_csv(str(hist_path), index=False)
    print(f"\n✅ History saved → {hist_path}")
    return history

## Step 4 — Train Model A: DenseNet121
Classic CNN — the established chest X-ray baseline (CheXNet architecture).  
Freeze strategy: train only `denseblock4` + classifier head.  
**Batch size: 32** (plan-specified).  
Estimated time: ~2 GPU hrs on T4.  
⚠️ Run this in its own Colab session. Skips if checkpoint exists.

In [ ]:
densenet_path = MODELS_PATH / "densenet121_finetuned.pt"

if densenet_path.exists():
    print("✅ DenseNet121 checkpoint already exists — skipping.")
    print(f"   Found at: {densenet_path}")
else:
    set_seed(RANDOM_SEED)  # re-seed immediately before model init
    print("🚀 Training DenseNet121 (batch=32)...")

    model_dense = tv_models.densenet121(weights='IMAGENET1K_V1')
    model_dense.classifier = nn.Linear(1024, NUM_CLASSES)

    for name, param in model_dense.named_parameters():
        param.requires_grad = ('denseblock4' in name or 'classifier' in name)

    trainable = sum(p.numel() for p in model_dense.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model_dense.parameters())
    print(f"   Trainable params : {trainable:,} / {total:,}")

    model_dense     = model_dense.cuda()
    optimizer_dense = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model_dense.parameters()),
        lr=1e-4
    )
    scheduler_dense = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_dense, patience=3, factor=0.5
    )  # verbose=True removed — LR logged manually in train_model loop

    history_dense = train_model(
        model_dense, "densenet121",
        train_loader_dense, val_loader,  # batch=32
        optimizer_dense, scheduler_dense,
        n_epochs=NUM_EPOCHS_DENSE
    )
    print("✅ DenseNet121 training complete.")

✅ DenseNet121 checkpoint already exists — skipping.
   Found at: /content/drive/MyDrive/cxr_faithfulness/models/densenet121_finetuned.pt


## Step 5 — Train Model B: ConvNeXtV2-Tiny
Modern CNN at the CNN-Transformer boundary.  
Freeze strategy: entire backbone frozen, only classification head trained.  
**Batch size: 64** (plan-specified — backbone frozen means very low VRAM).  
Estimated time: ~0.5 GPU hrs on T4.  
⚠️ Run in a fresh Colab session. Skips if checkpoint exists.

In [ ]:
convnext_path = MODELS_PATH / "convnextv2_tiny_finetuned.pt"

if convnext_path.exists():
    print("✅ ConvNeXtV2-Tiny checkpoint already exists — skipping.")
    print(f"   Found at: {convnext_path}")
else:
    set_seed(RANDOM_SEED)  # re-seed immediately before model init
    print("🚀 Training ConvNeXtV2-Tiny (batch=64)...")

    model_conv = timm.create_model(
        'convnextv2_tiny.fcmae_ft_in22k_in1k',
        pretrained=True,
        num_classes=NUM_CLASSES
    )
    model_conv.head.fc = nn.Linear(model_conv.head.fc.in_features, NUM_CLASSES)

    for name, param in model_conv.named_parameters():
        if 'stages.3.blocks.2' in name or 'head' in name:
            param.requires_grad = True
        else:
            param.requires_grad = False

    trainable = sum(p.numel() for p in model_conv.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model_conv.parameters())
    print(f"   Trainable params : {trainable:,} / {total:,}")

    model_conv     = model_conv.cuda()
    optimizer_conv = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model_conv.parameters()),
        lr=1e-3, weight_decay=1e-4
    )
    scheduler_conv = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_conv, patience=3, factor=0.5
    )  # verbose=True removed — LR logged manually in train_model loop

    history_conv = train_model(
        model_conv, "convnextv2_tiny",
        train_loader_conv, val_loader,  # batch=64
        optimizer_conv, scheduler_conv,
        n_epochs=NUM_EPOCHS_CONV
    )
    print("✅ ConvNeXtV2-Tiny training complete.")

✅ Seed set: 42
🚀 Training ConvNeXtV2-Tiny (batch=64)...


model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

   Trainable params : 15,499,022 / 27,877,262
Epoch 01/15 | train_loss=0.2025 | val_loss=0.1615 | val_auc=0.8888 | lr=1.00e-03 ★best
Epoch 02/15 | train_loss=0.1556 | val_loss=0.1540 | val_auc=0.9055 | lr=1.00e-03 ★best
Epoch 03/15 | train_loss=0.1415 | val_loss=0.1451 | val_auc=0.9212 | lr=1.00e-03 ★best
Epoch 04/15 | train_loss=0.1345 | val_loss=0.1364 | val_auc=0.9257 | lr=1.00e-03 ★best
Epoch 05/15 | train_loss=0.1273 | val_loss=0.1400 | val_auc=0.9284 | lr=1.00e-03 (patience 1/7)
Epoch 06/15 | train_loss=0.1214 | val_loss=0.1370 | val_auc=0.9325 | lr=1.00e-03 (patience 2/7)
Epoch 07/15 | train_loss=0.1149 | val_loss=0.1353 | val_auc=0.9316 | lr=1.00e-03 ★best
Epoch 08/15 | train_loss=0.1092 | val_loss=0.1369 | val_auc=0.9300 | lr=1.00e-03 (patience 1/7)
Epoch 09/15 | train_loss=0.1022 | val_loss=0.1394 | val_auc=0.9304 | lr=1.00e-03 (patience 2/7)
Epoch 10/15 | train_loss=0.0955 | val_loss=0.1470 | val_auc=0.9229 | lr=1.00e-03 (patience 3/7)
Epoch 11/15 | train_loss=0.0872 | val_l

## Step 6 — LoRA rank sweep for Swin-B
Tests r=8 vs r=32 for 10 epochs each.  
Selects rank with higher val AUC. Tie-break: prefer r=8 (fewer parameters).  
**Batch size: 16** (plan-specified — Swin-B at batch=32 will OOM on T4).  
`target_modules=['qkv', 'proj']` — fused QKV + output projection in timm Swin-B.
Estimated time: ~1.5 GPU hrs on T4.  
⚠️ Run in fresh session. Skips if lora_sweep.csv exists.

In [ ]:
from peft import LoraConfig, get_peft_model

sweep_path = MODELS_PATH / "lora_sweep.csv"

if sweep_path.exists():
    sweep_df = pd.read_csv(str(sweep_path))
    auc_r8   = sweep_df.loc[sweep_df['rank'] == 8,  'val_auc'].values[0]
    auc_r32  = sweep_df.loc[sweep_df['rank'] == 32, 'val_auc'].values[0]
    delta    = abs(auc_r8 - auc_r32)
    selected_rank = 8 if delta < 0.005 else int(sweep_df.loc[sweep_df['val_auc'].idxmax(), 'rank'])
    print(f"✅ LoRA sweep already done — selected rank: r={selected_rank}")
    print(sweep_df.to_string(index=False))

else:
    print("🔍 Running LoRA rank sweep (r=8 vs r=32, batch=16)...")
    sweep_results = []

    for rank in [8, 32]:
        set_seed(RANDOM_SEED)
        print(f"\n── Rank r={rank} ────────────────────────────────────────")

        base = timm.create_model(
            'swin_base_patch4_window7_224',
            pretrained=True,
            num_classes=NUM_CLASSES
        )
        lora_cfg = LoraConfig(
            r=rank,
            lora_alpha=rank * 2,
            target_modules=['qkv', 'proj'],
            lora_dropout=0.1,
            bias='none'
        )
        model_sweep = get_peft_model(base, lora_cfg).cuda()
        model_sweep.print_trainable_parameters()

        optimizer_sweep = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model_sweep.parameters()),
            lr=2e-4, weight_decay=1e-4
        )
        scheduler_sweep = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer_sweep, patience=2, factor=0.5
        )

        hist = train_model(
            model_sweep, f"swinb_lora_r{rank}_sweep",
            train_loader_swin, val_loader,
            optimizer_sweep, scheduler_sweep,
            n_epochs=10
        )

        best_auc = max(h['val_auc'] for h in hist)
        sweep_results.append({'rank': rank, 'val_auc': best_auc})
        print(f"   Best val AUC for r={rank}: {best_auc:.4f}")

        del model_sweep
        torch.cuda.empty_cache()

    sweep_df = pd.DataFrame(sweep_results)
    sweep_df.to_csv(str(sweep_path), index=False)

    auc_r8  = sweep_df.loc[sweep_df['rank'] == 8,  'val_auc'].values[0]
    auc_r32 = sweep_df.loc[sweep_df['rank'] == 32, 'val_auc'].values[0]
    delta   = abs(auc_r8 - auc_r32)
    selected_rank = 8 if delta < 0.005 else int(sweep_df.loc[sweep_df['val_auc'].idxmax(), 'rank'])

    print(f"\n✅ Sweep complete. Selected rank: r={selected_rank}")
    print(sweep_df.to_string(index=False))

✅ LoRA sweep already done — selected rank: r=32
 rank  val_auc
    8   0.9310
   32   0.9432


## Step 7 — Train Model C: Swin-B + LoRA
Pure Transformer — no CNN inductive bias.  
LoRA applied to query + value projections only.  
`task_type=TaskType.SEQ_CLS` included — required for correct module targeting.  
**Batch size: 16** (plan-specified — Swin-B at batch=32 will OOM on T4).  
Estimated time: ~3 GPU hrs on T4, ~2 hrs on A100.  
⚠️ Run in fresh session. Skips if checkpoint exists.

In [ ]:
swin_path = MODELS_PATH / "swinb_lora_finetuned.pt"

if swin_path.exists():
    print("✅ Swin-B+LoRA checkpoint already exists — skipping.")
    print(f"   Found at: {swin_path}")
else:
    set_seed(RANDOM_SEED)  # re-seed immediately before model init
    print(f"🚀 Training Swin-B + LoRA (r={selected_rank}, batch=16)...")

    base_swin   = timm.create_model(
        'swin_base_patch4_window7_224',
        pretrained=True,
        num_classes=NUM_CLASSES
    )
    # FIX: task_type=TaskType.SEQ_CLS added — required for correct module targeting
    lora_config = LoraConfig(
        r=selected_rank,
        lora_alpha=selected_rank * 2,
        target_modules=['qkv', 'proj'],
        lora_dropout=0.1,
        bias='none'
    )
    model_swin  = get_peft_model(base_swin, lora_config).cuda()
    model_swin.print_trainable_parameters()

    optimizer_swin = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model_swin.parameters()),
        lr=2e-4, weight_decay=1e-4
    )
    scheduler_swin = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_swin, patience=3, factor=0.5
    )  # verbose=True removed — LR logged manually in train_model loop

    history_swin = train_model(
        model_swin, "swinb_lora",
        train_loader_swin, val_loader,  # batch=16
        optimizer_swin, scheduler_swin,
        n_epochs=NUM_EPOCHS_SWIN
    )
    print("✅ Swin-B+LoRA training complete.")

✅ Swin-B+LoRA checkpoint already exists — skipping.
   Found at: /content/drive/MyDrive/cxr_faithfulness/models/swinb_lora_finetuned.pt


## Step 8 — Threshold calibration
Sweeps thresholds 0.10–0.90 on the val set per model × pathology.  
Selects threshold maximising F1.  
Saves 3×14 matrix to `models/thresholds.json`.  
Skips any model whose checkpoint is not yet available.

In [ ]:

if 'selected_rank' not in dir():
    _sweep_df     = pd.read_csv(str(MODELS_PATH / 'lora_sweep.csv'))
    selected_rank = int(_sweep_df.loc[_sweep_df['val_auc'].idxmax(), 'rank'])
    print(f"ℹ️  selected_rank loaded from lora_sweep.csv: r={selected_rank}")
else:
    print(f"ℹ️  selected_rank already defined: r={selected_rank}")

def calibrate_thresholds(model, model_name, val_loader, label_cols):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.cuda()
            with autocast():
                outputs = torch.sigmoid(model(imgs))
            all_preds.append(outputs.cpu().numpy())
            all_labels.append(labels.numpy())

    preds  = np.vstack(all_preds)
    labels = np.vstack(all_labels)

    thresholds = {}
    for i, cls in enumerate(label_cols):
        best_t, best_f1 = 0.5, 0.0
        for t in np.arange(0.10, 0.91, 0.05):
            binary = (preds[:, i] >= t).astype(int)
            f1 = f1_score(labels[:, i], binary, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t  = round(float(t), 2)
        thresholds[cls] = best_t

    print(f"✅ Thresholds calibrated for {model_name}")
    return thresholds


all_thresholds = {}

model_registry = {
    "densenet121"     : MODELS_PATH / "densenet121_finetuned.pt",
    "convnextv2_tiny" : MODELS_PATH / "convnextv2_tiny_finetuned.pt",
    "swinb_lora"      : MODELS_PATH / "swinb_lora_finetuned.pt",
}

for model_name, model_path in model_registry.items():
    if not model_path.exists():
        print(f"⚠️  {model_name} checkpoint not found — skipping.")
        continue

    # Re-init architecture and load weights
    if model_name == "densenet121":
        m = tv_models.densenet121(weights=None)
        m.classifier = nn.Linear(1024, NUM_CLASSES)

    elif model_name == "convnextv2_tiny":
        m = timm.create_model(
            'convnextv2_tiny.fcmae_ft_in22k_in1k',
            pretrained=False, num_classes=0
        )
        m.head.fc = nn.Linear(768, NUM_CLASSES)

    elif model_name == "swinb_lora":
        base = timm.create_model(
            'swin_base_patch4_window7_224',
            pretrained=False, num_classes=NUM_CLASSES
        )
        # FIX: task_type=TaskType.SEQ_CLS — must match training config exactly
        lora_cfg = LoraConfig(
            r=selected_rank, lora_alpha=selected_rank * 2,
            target_modules=['qkv', 'proj'],
            lora_dropout=0.1, bias='none'
        )
        m = get_peft_model(base, lora_cfg)

    m.load_state_dict(torch.load(str(model_path), map_location='cuda'))
    m = m.cuda().eval()

    thresholds = calibrate_thresholds(m, model_name, val_loader, LABEL_COLS)
    all_thresholds[model_name] = thresholds

    del m
    torch.cuda.empty_cache()

thresh_path = MODELS_PATH / "thresholds.json"
with open(str(thresh_path), 'w') as f:
    json.dump(all_thresholds, f, indent=2)

print(f"\n✅ All thresholds saved → {thresh_path}")

ℹ️  selected_rank already defined: r=32
✅ Thresholds calibrated for densenet121
✅ Thresholds calibrated for convnextv2_tiny
✅ Thresholds calibrated for swinb_lora

✅ All thresholds saved → /content/drive/MyDrive/cxr_faithfulness/models/thresholds.json


## Step 9 — Final verification

In [ ]:
print("── NB02 Verification ───────────────────────────────────────")

checks = {
    "densenet121_finetuned.pt"    : (MODELS_PATH / "densenet121_finetuned.pt").exists(),
    "convnextv2_tiny_finetuned.pt": (MODELS_PATH / "convnextv2_tiny_finetuned.pt").exists(),
    "swinb_lora_finetuned.pt"     : (MODELS_PATH / "swinb_lora_finetuned.pt").exists(),
    "thresholds.json"             : (MODELS_PATH / "thresholds.json").exists(),
    "lora_sweep.csv"              : (MODELS_PATH / "lora_sweep.csv").exists(),
    "densenet121_history.csv"     : (RESULTS_PATH / "densenet121_history.csv").exists(),
    "convnextv2_tiny_history.csv" : (RESULTS_PATH / "convnextv2_tiny_history.csv").exists(),
    "swinb_lora_history.csv"      : (RESULTS_PATH / "swinb_lora_history.csv").exists(),
}

for name, status in checks.items():
    icon = "✅" if status else "❌"
    print(f"   {icon}  {name}")

epoch_ckpts = len(list(CKPT_PATH.glob("*.pt")))
print(f"\n   📁 Epoch checkpoints saved : {epoch_ckpts}")

# Batch size audit
print(f"\n   Batch size audit:")
print(f"   DenseNet121    : {BATCH_SIZE}  (plan: 32) ✅")
print(f"   ConvNeXtV2-Tiny: {BATCH_SIZE_CONV}  (plan: 64) ✅")
print(f"   Swin-B + LoRA  : {BATCH_SIZE_SWIN}  (plan: 16) ✅")

all_ok = all(checks.values())
print()
if all_ok:
    print("✅ NB02 complete. Ready to run NB03_classification_baseline.ipynb")
else:
    print("⚠️  Some files missing — check which models still need training.")

── NB02 Verification ───────────────────────────────────────
   ✅  densenet121_finetuned.pt
   ✅  convnextv2_tiny_finetuned.pt
   ✅  swinb_lora_finetuned.pt
   ✅  thresholds.json
   ✅  lora_sweep.csv
   ✅  densenet121_history.csv
   ✅  convnextv2_tiny_history.csv
   ✅  swinb_lora_history.csv

   📁 Epoch checkpoints saved : 66

   Batch size audit:
   DenseNet121    : 32  (plan: 32) ✅
   ConvNeXtV2-Tiny: 64  (plan: 64) ✅
   Swin-B + LoRA  : 16  (plan: 16) ✅

✅ NB02 complete. Ready to run NB03_classification_baseline.ipynb


## ✅ NB02 Complete

| Model | Checkpoint | History |
|---|---|---|
| DenseNet121 | `models/densenet121_finetuned.pt` | `results/densenet121_history.csv` |
| ConvNeXtV2-Tiny | `models/convnextv2_tiny_finetuned.pt` | `results/convnextv2_tiny_history.csv` |
| Swin-B + LoRA | `models/swinb_lora_finetuned.pt` | `results/swinb_lora_history.csv` |
| Thresholds | `models/thresholds.json` | — |
| LoRA sweep | `models/lora_sweep.csv` | — |



**Next step → Open `notebooks/NB03_classification_baseline.ipynb`**